A folder namepattern as: gst_811s_vf_005_SOH080-00 is assumed. We use gst_811s_vf_005 as the cell name.  
Futhermore we extract vf as the electrolyte and everything after SOH as an SOH, here 80,00

In [ ]:
from zahner_analysis.file_import.ism_import import IsmImport
import glob
import os
import re
import pandas as pd
import numpy as np

In [ ]:
path_to_data = r"C:\Data\Daten\Zahner"

In [ ]:
file_list = glob.glob(path_to_data+"/**/*.ism", recursive=True)

In [ ]:
def read_file(current_file):
    folder_name = os.path.basename(os.path.dirname(current_file))
    (file_path, file_name) = os.path.split(current_file)
    zahner_file = IsmImport(current_file)

    cell_name = folder_name.split('_SOH')[0]
    cell_electrolyte = cell_name.split('_')[-2]
    cell_soh = float(folder_name.split('_SOH')[-1].split('_')[0].replace('-','.'))/100

    frequency = zahner_file.frequency
    z_abs = zahner_file.impedance
    z_phase = zahner_file.phase

    time = zahner_file.getMeasurementStartDateTime()

    pattern = re.compile('-?[0-9]*\.?[0-9]+[MU]?[av]')
    try:
        [voltage, excitation_voltage, excitation_current] = pattern.findall(zahner_file.getMetaData().decode('ascii', errors='ignore'))[:3]
    except:
        print(zahner_file.getMetaData().decode('ascii', errors='ignore'))
        print(pattern.findall(zahner_file.getMetaData().decode('ascii', errors='ignore')))

    def to_float(s):
        if s[-2] == "M":
            return abs(float(s[:-2])) * 1e-3
        elif s[-2] == "U":
            return abs(float(s[:-2])) * 1e-6
        else:
            return abs(float(s[:-1]))

    voltage = to_float(voltage)
    excitation_voltage = to_float(excitation_voltage)
    excitation_current = to_float(excitation_current)
    return pd.DataFrame({"Voltage": voltage, "excitation_voltage": excitation_voltage, "excitation_current": excitation_current, "cellname": cell_name, "Time": time, "cell_electrolyte": cell_electrolyte, "SOH": cell_soh, "EIS_Frequency": frequency, "EIS_Z_phase": z_phase, "EIS_Z_abs": z_abs})

df = pd.concat([read_file(f) for f in file_list])

def cell_to_file(df):
    df = df.sort_values("Time")
    times = np.array([str(x) for x in sorted(set(df["Time"].values))])
    timestrings = np.array([str(x) for x in df["Time"].values])
    measurement_idx = np.empty(len(df))
    for idx, t in enumerate(times):
        measurement_idx[timestrings == t] = idx
    cellname = df["cellname"].iloc[0]
    df["EIS_measurement_id"] = measurement_idx +1
    df["EIS_measurement_id"] = df["EIS_measurement_id"].astype(int)
    df = df.drop("cellname", axis=1)
    df.to_csv(f"{path_to_data}/export/{cellname}.csv", index=False)


df.groupby("cellname").apply(cell_to_file)